# Week 1: Exercise 3 - Basic Agent Loop

**Goal:** Implement the core loop that makes an agent an agent.

**This is the heart of every agent framework.**


In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv("../.env")

client = OpenAI(
    base_url="https://generativelanguage.googleapis.com/v1beta/openai",
    api_key=os.environ.get("GEMINI_API_KEY"),
)
print("API client ready")


API client ready


## Step 1: Define Tool Schemas
**TODO:** Define schemas for `get_time` and `add_numbers`.


In [2]:
TOOLS = [
    {
    "type": "function",
    "function": {
        "name": "get_time",
        "description": "Return current time as 'YYYY-MM-DD HH:MM:SS'.",
        "parameters": {
            "type": "object",
            "properties": {},
            "required": [],
        }
    },
    },
    {
    "type": "function",
    "function": {
        "name": "add_numbers",
        "description": "Add a and b together",
        "parameters": {
            "type": "object",
            "properties": {
                'a': {"type": "integer", "description": "Any number"},
                'b': {"type": "integer", "description": "Any number"}
            },
            "required": ['a','b'],
        }
    },
    }
]


## Step 2: Implement the Tool Functions


In [3]:
def get_time():
    """TODO: Return current time as 'YYYY-MM-DD HH:MM:SS'."""
    # Hint: from datetime import datetime
    #       datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    import datetime
    return datetime.datetime.now().strftime("%Y-%M-%D %H:%M:%S")


def add_numbers(a: int, b: int):
    """TODO: Return a + b."""
    return a + b


## Step 3: Implement the Dispatcher


In [4]:
def call_function(name, args):
    """TODO: Map function names to implementations."""
    # Hint: same pattern as exercise 2's call_function, with two tools
    func = {
        "get_time": get_time,
        "add_numbers": add_numbers
    }

    try:
        if name in func:
            return func[name](**args)
        else: raise ValueError(f"Error 404! Function {name} not found!")
    except TypeError as TE:
        return f"Error: {TE}"


## Step 4: Implement the Agent Loop

**TODO:** Implement `run_agent_loop`.

1. Build messages, loop up to max_iterations
2. Call LLM with tools, append response
3. If no tool_calls -> return the answer
4. Execute each tool, append results

**IMPORTANT:** Use `model_dump(exclude_none=True)` — Gemini rejects None values.


In [5]:
import json

def run_agent_loop(user_input, max_iterations=10):
    """TODO: Run the agent loop."""
    # Hint: messages = [system, user]
    #       for attempt in range(max_iterations):
    #           responses = client.chat.completions.create(
    #               model=os.environ.get("GEMINI_MODEL"),
    #               messages=messages, tools=TOOLS)
    #           respond = responses.choices[0].message
    #           messages.append(respond.model_dump(exclude_none=True))
    #           if not respond.tool_calls: return respond.content
    #           for tool in respond.tool_calls:
    #               func_name/func_args (json.loads!), call_function,
    #               append {"role": "tool", "tool_call_id": tool.id,
    #                       "name": func_name, "content": json.dumps(result)}
    messages = [{"role": "system","content":"You're an agent with tools!"},
                {"role": "user", "content": user_input}]
    
    for iter in range(max_iterations):
        response = client.chat.completions.create(
            model = os.environ.get("GEMINI_3.5_MODEL"),
            messages = messages,
            tools = TOOLS
        )

        message = response.choices[0].message
        messages.append(message.model_dump(exclude_none = True))

        if not message.tool_calls:
            return message.content

        for tool in message.tool_calls:
            func_name = tool.function.name
            func_args = json.loads(tool.function.arguments)

            func_response = call_function(func_name, func_args)

            messages.append({
                "role": "tool",
                "tool_call_id": tool.id,
                "name": func_name,
                "content": json.dumps(func_response)
            })


## Step 5: Test It


In [6]:
print(run_agent_loop("What time is it?"))


It is currently 11:00 AM.


In [7]:
print(run_agent_loop("What is 25 + 17?"))


25 + 17 is 42.


In [8]:
print(run_agent_loop("What time is it and what is 100 + 200?"))


The current time is 2026-01-08 11:01:07, and 100 + 200 is 300.


## Key Takeaways
- The loop is the agent
- Tool results go back as role: 'tool' messages
